In [12]:
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
import pymc as pm
import arviz as az
import pandas as pd
from collections import defaultdict, Counter
from dataclasses import dataclass, field
from enum import Enum
from typing import Optional
import random
import itertools
import warnings
import math

# Notebook config
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
%matplotlib inline

# Board Game Simulation

## Volcano Rush

### Abstract

*(To be filled at the end of the project)*

- Summary of simulation approach and scope
- Key findings: win rates, character balance, resource dynamics
- Main conclusions on game design

### Introduction

Game balance is considered critical to the success of any game, yet there is no universal consensus on what it actually means [1]. It can be broadly interpreted as the process of tuning a game's rules, difficulty, play time, resources, and role abilities so that no single strategy, character, or configuration dominates the experience. For multiplayer games in particular, balance has several dimensions: ensuring no starting position is inherently advantageous, preventing any strategy from being strictly dominant, and calibrating the cost-to-benefit ratio across different game objects [1].

As a semi-cooperative board game, *Volcano Rush* faces a specific balancing challenge: the group must share a win condition (escaping the volcano), while players also compete for individual scores. A well-balanced game therefore needs a group win rate that feels achievable but not trivial, and individual scores that are not dominated by a single character role. Compounding this, the game supports 4 to 8 players, meaning balance must hold across a wide range of player configurations.

As with most game design problems, there is no deterministic recipe for achieving balance. *Volcano Rush* is both strategic and social, and raw numbers can sometimes mislead. Simulation is not a substitute for playtesting, but it provides a principled starting point for identifying where imbalances may arise before bringing the game to the table.

### Related Work

#### Monte Carlo simulation

Monte Carlo simulation estimates probabilities by running many random trials instead of solving mathematical formulas, which can be extremely difficult for complex games. Running $n$ simulated games under a given strategy $M$ yields an empirical win probability:

We define an indicator variable for each simulation:

- $X_i = 1$ if simulation $i$ results in a win  
- $X_i = 0$ otherwise  

The empirical win probability after $n$ simulations is:

$$
\hat{p}_n(M) = \frac{1}{n} \sum_{i=1}^{n} X_i
$$

As the number of simulations increases, this estimate converges to the true win probability:

$$
p(M) = \lim_{n \to \infty} \frac{1}{n} \sum_{i=1}^{n} X_i
$$

This follows from the **Law of Large Numbers**, which states that the sample average converges to the expected value (true probability) as the number of trials grows. This makes Monte Carlo methods a good fit for games with many possible situations, where computing the exact answer is not practical.

#### Monte Carlo Tree Search (MCTS)

Monte Carlo methods are also widely used in game AI through Monte Carlo Tree Search (MCTS) [2], which builds a tree of possible future game states and tests them with simulations. 
At each step, the algorithm faces a trade-off between two ideas:
- *exploitation* - choosing moves that have already shown good results  
- *exploration* - trying moves that have not been tested much but might be better  

This balance is handled by the **Upper Confidence Bound for Trees** (UCT) formula:

$$ \mathrm{UCT} = \frac{W_i}{N_i} + c \sqrt{\frac{\ln N_p}{N_i}} $$

The formula combines two parts:
- $\frac{W_i}{N_i}$ - the **exploitation term**, which represents how successful a move has been so far (its average win rate)
- $c \sqrt{\frac{\ln N_p}{N_i}}$ - the **exploration term**, which gives a higher value to moves that have been tried fewer times, encouraging the algorithm to explore them
- $c$ - a constant that controls the balance between exploration and exploitation (higher values lead to more exploration)

In practice, MCTS does not explore all possible game paths. Instead, it gradually builds the search tree through repeated simulations. At the beginning, it gathers initial data by trying different possible moves. As more simulations are performed, the algorithm starts to focus more on the moves that show better results, exploring them in greater depth. However, less-explored moves are still occasionally tested due to the exploration term in the UCT formula, ensuring that potentially good alternatives are not missed.

In this work, however, we use simpler repeated simulations with rule-based agents to study game balance rather than to improve decision-making [2].

#### Agent-Based Modeling (ABM)

In Agent-Based Modeling (ABM), each agent follows a set of simple rules and interacts with other agents and the world around it [3]. Over time, these small interactions can produce complex group behaviors that would be hard to predict just by looking at the rules alone.

Compared to pure Monte Carlo simulation, ABM allows for more realistic behavior. Monte Carlo is fully random, but in ABM, agents can follow rules that reflect real human decisions. This approach has been used in research: Deadman, Schlager, and Gimblett (2000) interviewed participants after a game experiment and used their answers to design the decision rules for ABM agents [5]. For *Volcano Rush*, we could do the same - interview players after a playtest session and use what we learn to make the agents behave more like real people [4].

Traditional game theory assumes that players always make the best possible decision to get the best outcome for themselves. But in real life, people are often influenced by emotions, habits, and not having all the information [3]. This means game theory models do not always reflect how real people actually play.

### Data

This study uses no external dataset. All data is generated by the simulation itself - each run of the simulation produces one game record. The input to the simulation is a fixed set of game parameters taken directly from the game rules.

#### Game Parameters

The resource deck contains **60 cards** split equally between three types: Wood, Stone, and Rope (20 each). The deck is reshuffled and reused when it runs out. The camp has two **shared tools** - Knife and Vessel - which can become damaged and must be repaired before use.

There are **6 character roles**: Builder, Fire Starter, Craftsman, Cook, Gatherer, and Sailor. Each has a unique ability that affects resource requirements, point rewards, or complication handling. For games of 4 or 5 players, the Craftsman role is always assigned and the remaining roles are drawn randomly. For 6 players, all six roles are used once. For 7–8 players, one or two roles are repeated.

The **mission pool** consists of 8 standard missions and up to 5 boat missions. At any time, exactly 3 missions are active and new missions are drawn when one is completed or discarded. The number of boat missions in play - and the number of boat parts needed to win - depends on player count:

| Players | Boat Parts Required | Active Boat Missions |
|---|---|---|
| 4–5 | 2 | 2 randomly selected |
| 6 | 4 | All 4 base missions |
| 7–8 | 5 | All 4 base missions + Fit the Rudder |

The **Complication deck** contains 11 cards (including 2 neutral cards with no effect) and is reshuffled when exhausted. The **Volcano deck** contains 11 penalty cards with the Eruption card fixed at the bottom - when it is drawn, the game ends in a loss.
Card draws from the resource, complication, and volcano decks are modeled as stochastic processes with uniform random sampling.

#### Simulation Assumptions

Where the game rules leave room for interpretation, the following assumptions are applied:

| Rule Area | Assumption |
|---|---|
| Exhaustion | Applies to all participants after resolution, regardless of success or failure |
| Craftsman repair | Starts round N, tool unavailable round N+1, available again round N+2 |
| Sailor - lesser evil | The less severe Complication card is chosen using a severity scoring function |
| Participant count | Treated as an exact requirement, no additional participants allowed |
| Damaged tool | A mission requiring a damaged tool automatically fails |
| Night Anxiety | Requires 1 non-participant helper; if none available, mission fails |
| Gather - default | Any player not participating automatically gathers |
| Gather - non-Gatherer | Drawing 1 resource via Gather does not cause Exhaustion |
| Gather - Gatherer role | Using the boosted draw (2 resources) causes Exhaustion |

#### Output Variables

Each simulation run records: game outcome (win/loss), number of rounds played, individual score per player, win rate per role (role dominance), number of boat parts completed, and number of volcano cards remaining when the game ends.

### Methods

The simulation runs each game as a sequence of rounds following the 6-step loop described in `docs/game_rules.md`. Thousands of games are run per configuration, and outcomes are recorded to estimate win rates and individual scores empirically.

#### Agent Strategy

Each non-exhausted player agent follows a set of rule-based heuristics that approximate plausible human behavior. The rules apply in priority order each round:

**Mission voting (before the round starts)**
- Each agent votes for the mission where their character role has a direct ability bonus (e.g. Builder votes for Wood-heavy missions, Cook votes for Vessel missions, Fire Starter votes for "Light a Fire" or "Torch"). The mission with the most votes is selected.
- If no agent has a preference, a mission is chosen at random from the 3 active ones.

**Action decision (participate or gather)**
1. **Participate** - if the agent is not Exhausted and has enough resources to contribute the required amount and still keep at least 1 card in hand.
2. **Gather** - if the agent cannot or chooses not to participate.
   - **Gatherer role**: uses the boosted draw (2 resources) when not Exhausted and resources are below a threshold; otherwise gathers normally.

**Craftsman repair rule**
- If the Craftsman is not Exhausted and at least 1 tool is damaged, they skip the mission and start a repair instead.

**Boat mission priority**
- When the volcano deck has 4 or fewer cards remaining, all agents give maximum priority to boat missions in their voting.

**Low-pressure rules (only apply when volcano deck has more than 4 cards remaining)**
- *Resource conservation*: agents with only 1 card in hand always gather instead of participating.
- *Exhaustion spreading*: if more than half the team is Exhausted, non-Exhausted agents prefer to gather to rebuild resources for the next round.

#### Simulation Scenarios

Games are run across all player counts (4-8). Character assignment follows the game rules: Craftsman is always assigned in 4-5 player games; all 6 roles are used exactly once in 6-player games; repeated roles fill the extra slots in 7-8 player games. Each scenario runs N = 1,000-10,000 games. Sensitivity analysis varies the participation threshold and boat mission priority trigger.

#### Statistical Analysis

Win rates per scenario are estimated using a Bayesian Beta-Binomial model (PyMC). Posterior distributions are compared across player counts and character compositions using ArviZ credible intervals.

#### Define Game State & Parameters

In [2]:
class Character(Enum):
    BUILDER       = "builder"
    FIRE_STARTER  = "fire_starter"
    CRAFTSMAN     = "craftsman"
    COOK          = "cook"
    GATHERER      = "gatherer"
    SAILOR        = "sailor"


class Resource(Enum):
    WOOD  = "wood"
    STONE = "stone"
    ROPE  = "rope"


class Tool(Enum):
    KNIFE  = "knife"
    VESSEL = "vessel"


class PlayerAction(Enum):
    PARTICIPATE = "participate"
    GATHER      = "gather"
    REPAIR      = "repair"


class MissionType(Enum):
    FIRE    = "fire"
    SHELTER = "shelter"
    FOOD    = "food"
    BOAT    = "boat"


class MissionName(Enum):
    LIGHT_A_FIRE        = "light_a_fire"
    TORCH_FOR_THE_NIGHT = "torch_for_the_night"
    FETCH_WATER         = "fetch_water"
    BUILD_A_SHELTER     = "build_a_shelter"
    FORTIFY_THE_CAMP    = "fortify_the_camp"
    GATHER_MATERIALS    = "gather_materials"
    HUNT                = "hunt"
    PREPARE_FOOD        = "prepare_food"
    CUT_THE_KEEL        = "cut_the_keel"
    ASSEMBLE_THE_HULL   = "assemble_the_hull"
    RAISE_THE_MAST      = "raise_the_mast"
    MAKE_THE_SAIL       = "make_the_sail"
    FIT_THE_RUDDER      = "fit_the_rudder"


class ComplicationCardName(Enum):
    MOSQUITO_ATTACK = "mosquito_attack"
    WET_WOOD        = "wet_wood"
    COLLAPSED_PATH  = "collapsed_path"
    SLIPPERY_ROCKS  = "slippery_rocks"
    BLUNT_BLADE     = "blunt_blade"
    CRACKED_VESSEL  = "cracked_vessel"
    HEAT_AND_THIRST = "heat_and_thirst"
    NIGHT_ANXIETY   = "night_anxiety"
    CAMP_PANIC      = "camp_panic"
    CALM_BREEZE     = "calm_breeze"
    CLEAR_SKY       = "clear_sky"


class VolcanoCardName(Enum):
    RAIN_AND_MUD   = "rain_and_mud"
    ASH_IN_THE_AIR = "ash_in_the_air"
    TREMOR         = "tremor"
    STORM          = "storm"
    LAVA_FLOW      = "lava_flow"
    PANIC          = "panic"
    COLLAPSE       = "collapse"
    HEAT_WAVE      = "heat_wave"
    SMOKE          = "smoke"
    EARTHQUAKE     = "earthquake"
    ERUPTION       = "eruption"

In [3]:
@dataclass(frozen = True)
class BonusEffect:
    resource_discount:      dict[Resource, int]       = field(default_factory = dict)
    resource_discount_any:  int                       = 0
    skip_next_complication: bool                      = False
    gather_bonus:           int                       = 0
    negates_volcano_card:   Optional[VolcanoCardName] = None
    repair_tool:            bool                      = False
    no_exhaustion:          bool                      = False
    protect_next_failure:   bool                      = False
    boat_part:              bool                      = False


@dataclass(frozen = True)
class Mission:
    name:               MissionName
    required_resources: dict[Resource, int]
    players_count:      int
    points:             int
    mission_type:       MissionType
    required_tools:     list[Tool]            = field(default_factory = list)
    bonus_on_success:   Optional[BonusEffect] = None


MISSIONS: dict[MissionName, Mission] = {
    # Fire missions
    MissionName.LIGHT_A_FIRE: Mission(
        name               = MissionName.LIGHT_A_FIRE,
        required_resources = {Resource.WOOD: 2, Resource.STONE: 1},
        players_count      = 2,
        points             = 2,
        mission_type       = MissionType.FIRE,
        bonus_on_success   = BonusEffect(resource_discount = {Resource.WOOD: 1}),
    ),
    MissionName.TORCH_FOR_THE_NIGHT: Mission(
        name               = MissionName.TORCH_FOR_THE_NIGHT,
        required_resources = {Resource.WOOD: 2, Resource.ROPE: 1},
        players_count      = 3,
        points             = 3,
        mission_type       = MissionType.FIRE,
        bonus_on_success   = BonusEffect(skip_next_complication = True),
    ),
    # Food missions
    MissionName.FETCH_WATER: Mission(
        name               = MissionName.FETCH_WATER,
        required_resources = {Resource.ROPE: 2, Resource.WOOD: 1},
        players_count      = 3,
        points             = 3,
        mission_type       = MissionType.FOOD,
        required_tools     = [Tool.VESSEL],
        bonus_on_success   = BonusEffect(gather_bonus = 1),
    ),
    MissionName.HUNT: Mission(
        name               = MissionName.HUNT,
        required_resources = {Resource.STONE: 2, Resource.ROPE: 2},
        players_count      = 3,
        points             = 3,
        mission_type       = MissionType.FOOD,
        required_tools     = [Tool.KNIFE],
        bonus_on_success   = BonusEffect(protect_next_failure = True),
    ),
    MissionName.PREPARE_FOOD: Mission(
        name               = MissionName.PREPARE_FOOD,
        required_resources = {Resource.WOOD: 1, Resource.STONE: 1},
        players_count      = 3,
        points             = 3,
        mission_type       = MissionType.FOOD,
        required_tools     = [Tool.KNIFE, Tool.VESSEL],
        bonus_on_success   = BonusEffect(no_exhaustion = True),
    ),
    # Shelter missions
    MissionName.BUILD_A_SHELTER: Mission(
        name               = MissionName.BUILD_A_SHELTER,
        required_resources = {Resource.WOOD: 2, Resource.ROPE: 1, Resource.STONE: 1},
        players_count      = 4,
        points             = 4,
        mission_type       = MissionType.SHELTER,
        required_tools     = [Tool.KNIFE],
        bonus_on_success   = BonusEffect(negates_volcano_card = VolcanoCardName.RAIN_AND_MUD),
    ),
    MissionName.FORTIFY_THE_CAMP: Mission(
        name               = MissionName.FORTIFY_THE_CAMP,
        required_resources = {Resource.ROPE: 2, Resource.WOOD: 1},
        players_count      = 2,
        points             = 2,
        mission_type       = MissionType.SHELTER,
        required_tools     = [Tool.KNIFE],
        bonus_on_success   = BonusEffect(resource_discount_any = 1),
    ),
    MissionName.GATHER_MATERIALS: Mission(
        name               = MissionName.GATHER_MATERIALS,
        required_resources = {Resource.WOOD: 1, Resource.STONE: 2, Resource.ROPE: 2},
        players_count      = 3,
        points             = 2,
        mission_type       = MissionType.SHELTER,
        bonus_on_success   = BonusEffect(repair_tool = True),
    ),
    # Boat missions
    MissionName.CUT_THE_KEEL: Mission(
        name               = MissionName.CUT_THE_KEEL,
        required_resources = {Resource.WOOD: 3, Resource.ROPE: 2},
        players_count      = 3,
        points             = 1,
        mission_type       = MissionType.BOAT,
        required_tools     = [Tool.KNIFE],
        bonus_on_success   = BonusEffect(boat_part = True),
    ),
    MissionName.ASSEMBLE_THE_HULL: Mission(
        name               = MissionName.ASSEMBLE_THE_HULL,
        required_resources = {Resource.WOOD: 2, Resource.STONE: 2},
        players_count      = 3,
        points             = 1,
        mission_type       = MissionType.BOAT,
        bonus_on_success   = BonusEffect(boat_part = True),
    ),
    MissionName.RAISE_THE_MAST: Mission(
        name               = MissionName.RAISE_THE_MAST,
        required_resources = {Resource.WOOD: 2, Resource.ROPE: 2},
        players_count      = 3,
        points             = 1,
        mission_type       = MissionType.BOAT,
        required_tools     = [Tool.VESSEL],
        bonus_on_success   = BonusEffect(boat_part = True),
    ),
    MissionName.MAKE_THE_SAIL: Mission(
        name               = MissionName.MAKE_THE_SAIL,
        required_resources = {Resource.WOOD: 2, Resource.ROPE: 3},
        players_count      = 2,
        points             = 1,
        mission_type       = MissionType.BOAT,
        bonus_on_success   = BonusEffect(boat_part = True),
    ),
    MissionName.FIT_THE_RUDDER: Mission(
        name               = MissionName.FIT_THE_RUDDER,
        required_resources = {Resource.WOOD: 3, Resource.STONE: 1, Resource.ROPE: 3},
        players_count      = 4,
        points             = 1,
        mission_type       = MissionType.BOAT,
        required_tools     = [Tool.VESSEL],
        bonus_on_success   = BonusEffect(boat_part = True),
    ),
}

In [4]:
@dataclass(frozen = True)
class ComplicationCard:
    name:                    ComplicationCardName
    severity:                int
    extra_resources:         dict[Resource, int]  = field(default_factory = dict)
    extra_resources_any:     int                  = 0
    extra_per_participant:   int                  = 0
    damages_tool_on_success: Optional[Tool]       = None
    requires_extra_helper:   bool                 = False
    max_resource_per_type:   Optional[int]        = None
    conditional_on_resource: Optional[Resource]   = None


COMPLICATION_CARDS: dict[ComplicationCardName, ComplicationCard] = {
    ComplicationCardName.MOSQUITO_ATTACK: ComplicationCard(
        name            = ComplicationCardName.MOSQUITO_ATTACK,
        severity        = 1,
        extra_resources = {Resource.ROPE: 1},
    ),
    ComplicationCardName.WET_WOOD: ComplicationCard(
        name                    = ComplicationCardName.WET_WOOD,
        severity                = 1,
        extra_resources         = {Resource.WOOD: 1},
        conditional_on_resource = Resource.WOOD,
    ),
    ComplicationCardName.COLLAPSED_PATH: ComplicationCard(
        name            = ComplicationCardName.COLLAPSED_PATH,
        severity        = 1,
        extra_resources = {Resource.STONE: 1},
    ),
    ComplicationCardName.SLIPPERY_ROCKS: ComplicationCard(
        name                = ComplicationCardName.SLIPPERY_ROCKS,
        severity            = 3,
        extra_resources_any = 2,
    ),
    ComplicationCardName.BLUNT_BLADE: ComplicationCard(
        name                    = ComplicationCardName.BLUNT_BLADE,
        severity                = 2,
        damages_tool_on_success = Tool.KNIFE,
    ),
    ComplicationCardName.CRACKED_VESSEL: ComplicationCard(
        name                    = ComplicationCardName.CRACKED_VESSEL,
        severity                = 2,
        damages_tool_on_success = Tool.VESSEL,
    ),
    ComplicationCardName.HEAT_AND_THIRST: ComplicationCard(
        name                  = ComplicationCardName.HEAT_AND_THIRST,
        severity              = 4,
        extra_per_participant = 1,
    ),
    ComplicationCardName.NIGHT_ANXIETY: ComplicationCard(
        name                  = ComplicationCardName.NIGHT_ANXIETY,
        severity              = 5,
        requires_extra_helper = True,
    ),
    ComplicationCardName.CAMP_PANIC: ComplicationCard(
        name                  = ComplicationCardName.CAMP_PANIC,
        severity              = 3,
        max_resource_per_type = 1,
    ),
    ComplicationCardName.CALM_BREEZE: ComplicationCard(
        name     = ComplicationCardName.CALM_BREEZE,
        severity = 0,
    ),
    ComplicationCardName.CLEAR_SKY: ComplicationCard(
        name     = ComplicationCardName.CLEAR_SKY,
        severity = 0,
    ),
}

In [5]:
@dataclass(frozen = True)
class VolcanoCard:
    name:                        VolcanoCardName
    extra_resources:             dict[Resource, int]  = field(default_factory = dict)
    conditional_on_resource:     Optional[Resource]   = None
    extra_exhaustion_rounds:     int                  = 0
    discard_mission:             bool                 = False
    each_player_loses_resources: int                  = 0
    max_mission_participants:    Optional[int]        = None
    rich_player_loses_threshold: Optional[int]        = None
    gather_yields_zero:          bool                 = False
    mission_point_penalty:       int                  = 0
    extend_exhaustion_rounds:    int                  = 0
    is_eruption:                 bool                 = False


VOLCANO_CARDS: dict[VolcanoCardName, VolcanoCard] = {
    VolcanoCardName.RAIN_AND_MUD: VolcanoCard(name = VolcanoCardName.RAIN_AND_MUD, extra_resources = {Resource.WOOD: 2}, conditional_on_resource = Resource.WOOD),
    VolcanoCardName.ASH_IN_THE_AIR: VolcanoCard(name = VolcanoCardName.ASH_IN_THE_AIR, extra_exhaustion_rounds = 1),
    VolcanoCardName.TREMOR: VolcanoCard(name = VolcanoCardName.TREMOR, discard_mission = True),
    VolcanoCardName.STORM: VolcanoCard(name = VolcanoCardName.STORM, each_player_loses_resources = 1),
    VolcanoCardName.LAVA_FLOW: VolcanoCard(name = VolcanoCardName.LAVA_FLOW, extra_resources = {Resource.ROPE: 1}),
    VolcanoCardName.PANIC: VolcanoCard(name = VolcanoCardName.PANIC, max_mission_participants = 3),
    VolcanoCardName.COLLAPSE: VolcanoCard(name = VolcanoCardName.COLLAPSE, each_player_loses_resources = 1, rich_player_loses_threshold = 3),
    VolcanoCardName.HEAT_WAVE: VolcanoCard(name = VolcanoCardName.HEAT_WAVE, gather_yields_zero = True),
    VolcanoCardName.SMOKE: VolcanoCard(name = VolcanoCardName.SMOKE, mission_point_penalty = 1),
    VolcanoCardName.EARTHQUAKE: VolcanoCard(name = VolcanoCardName.EARTHQUAKE, extend_exhaustion_rounds = 1),
    VolcanoCardName.ERUPTION: VolcanoCard(name = VolcanoCardName.ERUPTION, is_eruption = True),
}

In [7]:
@dataclass
class ToolState:
    damaged:    bool          = False
    repair_due: Optional[int] = None


@dataclass
class Player:
    character:       Character
    resources:       list[Resource]  = field(default_factory = list)
    is_exhausted:    bool            = False
    exhausted_until: int             = 0
    score:           int             = 0


@dataclass
class GameState:
    players:                list[Player]
    active_missions:        list[MissionName]
    resource_deck:          list[Resource]
    complication_deck:      list[ComplicationCardName]
    volcano_deck:           list[VolcanoCardName]
    tools:                  dict[Tool, ToolState]
    boat_parts_required:    int
    boat_parts_built:       set[MissionName]            = field(default_factory = set)
    mission_pool:           list[MissionName]           = field(default_factory = list)
    round:                  int                         = 0
    skip_next_complication: bool                        = False
    protect_next_failure:   bool                        = False
    pending_volcano_card:   Optional[VolcanoCardName]   = None
    pending_bonus:          Optional[BonusEffect]       = None

#### Implement Simulation Engine

- Round loop: 6 steps (mission selection → participation → complication draw → resolution → exhaustion → mission refresh)
- Mission resolution: count participants, check tool availability, apply character abilities, check success
- Handle special complications (Night Anxiety requires a non-participant helper, Camp Panic limits resources per player, etc.)
- Handle volcano effects (Rain and Mud cost increases, Heat Wave extended exhaustion, Lava Flow removes missions, etc.)
- Terminal conditions: Eruption card drawn (loss) or all required boat missions complete (win)

In [8]:
INITIAL_RESOURCES_PER_PLAYER = 3
URGENT_VOLCANO_THRESHOLD = 4
DECK_RESOURCE_COUNT = 20


# ── Initialization ─────────────────────────────────────────────────────────

def prepare_resource_deck() -> list[Resource]:
    resource_deck = ([Resource.WOOD] * DECK_RESOURCE_COUNT + [Resource.STONE] * DECK_RESOURCE_COUNT + [Resource.ROPE] * DECK_RESOURCE_COUNT)
    random.shuffle(resource_deck)
    
    return resource_deck


def assign_characters(player_count: int) -> list[Character]:
    required_character = Character.CRAFTSMAN
    all_characters = list(Character)
    all_characters_count = len(all_characters)

    if player_count < all_characters_count:
        others = [c for c in all_characters if c != required_character]
        chosen = random.sample(others, player_count - 1)
        characters = [required_character] + chosen
    elif player_count == all_characters_count:
        characters = all_characters
    else:
        base_characters = random.sample(all_characters, all_characters_count)
        extras = random.sample(all_characters, player_count - all_characters_count)
        characters = base_characters + extras
    
    random.shuffle(characters)
    
    return characters


def prepare_players(player_count: int, resource_deck: list[Resource]) -> list[Player]:
    characters = assign_characters(player_count)

    players = []
    for char in characters:
        player = Player(character = char)
        player.resources.extend(resource_deck.pop() for _ in range(INITIAL_RESOURCES_PER_PLAYER))
        players.append(player)
    
    return players


def get_boat_missions(player_count: int) -> list[MissionName]:
    all_characters_count = len(list(Character))
    extra_mission = MissionName.FIT_THE_RUDDER

    base_boat_missions = [n for n, m in MISSIONS.items()
            if m.mission_type == MissionType.BOAT and n != extra_mission]
    
    if player_count < all_characters_count:
        return random.sample(base_boat_missions, 2)
    elif player_count == all_characters_count:
        return list(base_boat_missions)
    else:
        return list(base_boat_missions) + [extra_mission]


def get_mission_pool(player_count: int) -> list[MissionName]:
    boat_missions = get_boat_missions(player_count)
    standard_missions = [n for n, m in MISSIONS.items() if m.mission_type != MissionType.BOAT]
    mission_pool = standard_missions + boat_missions
    
    random.shuffle(mission_pool)
    
    return mission_pool


def prepare_complication_deck() -> list[ComplicationCardName]:
    compilation_deck = list(ComplicationCardName)
    random.shuffle(compilation_deck)
    
    return compilation_deck


def prepare_volcano_deck() -> list[VolcanoCardName]:
    eruption_card = VolcanoCardName.ERUPTION

    non_eruption = [v for v in VolcanoCardName if v != eruption_card]
    random.shuffle(non_eruption)
    
    volcano_deck = [eruption_card] + non_eruption

    return volcano_deck


def init_game(player_count: int) -> GameState:
    resource_deck = prepare_resource_deck()

    players = prepare_players(player_count, resource_deck)

    mission_pool = get_mission_pool(player_count)
    active_missions = [mission_pool.pop() for _ in range(3)]

    volcano_deck = prepare_volcano_deck()
    complication_deck = prepare_complication_deck()

    tools = {tool: ToolState() for tool in Tool}

    boat_missions = get_boat_missions(player_count)
    boat_parts_required = len(boat_missions)

    return GameState(
        players              = players,
        active_missions      = active_missions,
        resource_deck        = resource_deck,
        complication_deck    = complication_deck,
        volcano_deck         = volcano_deck,
        tools                = tools,
        boat_parts_required  = boat_parts_required,
        mission_pool         = mission_pool,
    )

In [ ]:
# ── Deck management ────────────────────────────────────────────────────────

def draw_resource(state: GameState) -> Resource:
    if not state.resource_deck:
        state.resource_deck = prepare_resource_deck()

    return state.resource_deck.pop()


def draw_complication(state: GameState) -> ComplicationCardName:
    return np.random.choice(state.complication_deck)


def draw_volcano(state: GameState) -> VolcanoCardName:
    return state.volcano_deck.pop()


def draw_mission(state: GameState) -> Optional[MissionName]:
    return state.mission_pool.pop() if state.mission_pool else None

In [10]:
# ── Exhaustion & repair ────────────────────────────────────────────────────

def refresh_exhaustion(state: GameState) -> None:
    for player in state.players:
        player.is_exhausted = state.round <= player.exhausted_until


def apply_exhaustion(players: list[Player], current_round: int, extra_rounds: int = 0) -> None:
    for player in players:
        player.exhausted_until = current_round + 1 + extra_rounds
        player.is_exhausted = True


def update_tool_repairs(state: GameState) -> None:
    for tool_state in state.tools.values():
        if tool_state.repair_due is not None and tool_state.repair_due <= state.round:
            tool_state.damaged = False
            tool_state.repair_due = None

In [ ]:
# ── Resource requirement calculation ───────────────────────────────────────

@dataclass(frozen = True)
class MissionRequirement:
    typed:     dict[Resource, int]
    any_extra: int


def compute_requirements(
    mission:      Mission,
    participants: list[Player],
    complication: ComplicationCard,
    state:        GameState,
) -> MissionRequirement:
    """
    Compute the total mission requirements for a mission attempt.

    Applies all active modifiers in order:
    1. Base mission requirements
    2. Pending volcano card extra resources
    3. Complication card extra resources and per-participant surcharges
    4. Pending bonus discounts and character ability discounts

    Mission requirements are clamped to zero — discounts cannot go negative.

    Args:
        mission:      The mission being attempted.
        participants: Players participating in the mission.
        complication: The complication card drawn for this attempt.
        state:        Current game state (pending bonus, pending volcano card, etc.).

    Returns:
        Mission requirements with typed resource costs and a wildcard any_extra count.
        typed resources must be paid with the exact resource type;
        any_extra may be paid with any resource type.
    """
    resource_requirements = dict(mission.required_resources)
    any_extra = 0

    # Apply pending volcano card extra resources
    if state.pending_volcano_card is not None:
        volcano_card = VOLCANO_CARDS[state.pending_volcano_card]
        for resource, amount in volcano_card.extra_resources.items():
            if volcano_card.conditional_on_resource is None or volcano_card.conditional_on_resource in resource_requirements:
                resource_requirements[resource] = resource_requirements.get(resource, 0) + amount

    # Apply complication card extras
    for resource, amount in complication.extra_resources.items():
        if complication.conditional_on_resource is None or complication.conditional_on_resource in resource_requirements:
            resource_requirements[resource] = resource_requirements.get(resource, 0) + amount
    any_extra += complication.extra_resources_any
    any_extra += complication.extra_per_participant * len(participants)

    # Apply pending bonus discounts and character ability discounts
    if state.pending_bonus is not None:
        bonus = state.pending_bonus
        for resource, amount in bonus.resource_discount.items():
            resource_requirements[resource] = resource_requirements.get(resource, 0) - amount
        any_extra -= bonus.resource_discount_any

    for player in participants:
        if player.character == Character.BUILDER:
            if resource_requirements.get(Resource.WOOD, 0) >= 2:
                resource_requirements[Resource.WOOD] = resource_requirements[Resource.WOOD] - 1
        elif player.character == Character.FIRE_STARTER:
            if mission.mission_type == MissionType.FIRE:
                any_extra -= 1

    # Clamp
    resource_requirements = {resource: max(0, value) for resource, value in resource_requirements.items()}
    any_extra = max(0, any_extra)
    
    return MissionRequirement(typed = resource_requirements, any_extra = any_extra)

In [ ]:
# ── Contribution check & resource deduction ────────────────────────────────

def check_and_contribute(
    participants:  list[Player],
    requirements:  MissionRequirement,
    max_per_type:  Optional[int],
) -> bool:
    """
    Check if participants can meet the typed and any_extra resource requirements,
    respecting max_per_type limits, and if so, deduct the resources from their hands.

    Args:
        participants: Players attempting the mission.
        requirements: Mission requirements with typed resource costs and a wildcard any_extra count.
        max_per_type: If set, limits the number of resources of the same type that can be
                      contributed per player (e.g. Camp Panic cap).

    Returns:
        True if requirements can be met and resources were deducted,
        False otherwise (no deduction on failure).
    """
    # Build available pool per resource (respecting Camp Panic cap)
    available: dict[Resource, int] = {}
    for player in participants:
        resources_by_type = Counter(player.resources)
        for resource, count in resources_by_type.items():
            if max_per_type is not None:
                count = min(count, max_per_type)
            available[resource] = available.get(resource, 0) + count

    # Net surplus per resource after subtracting typed requirements.
    # Union of both key sets ensures required-but-absent resources yield a negative entry.
    surplus_by_resource = {
        resource: available.get(resource, 0) - requirements.typed.get(resource, 0)
        for resource in set(available) | set(requirements.typed)
    }

    # Negative surplus means a typed requirement is unmet — return without deducting
    if any(surplus < 0 for surplus in surplus_by_resource.values()):
        return False

    # Total surplus must cover the wildcard any_extra requirement
    if sum(surplus_by_resource.values()) < requirements.any_extra:
        return False

    # Deduct typed requirements, then fill any_extra greedily (prefer most-abundant surplus)
    to_remove: dict[Resource, int] = dict(requirements.typed)
    remaining_any = requirements.any_extra
    for resource in sorted(surplus_by_resource, key = lambda r: -surplus_by_resource[r]):
        take = min(remaining_any, surplus_by_resource[resource])
        to_remove[resource] = to_remove.get(resource, 0) + take
        remaining_any -= take
        if remaining_any <= 0:
            break

    # Remove from player hands
    for resource, total in to_remove.items():
        left = total
        for player in participants:
            while left > 0 and resource in player.resources:
                player.resources.remove(resource)
                left -= 1

    return True

In [ ]:
# ── Mission resolution ─────────────────────────────────────────────────────

def resolve_mission(
    state:        GameState,
    mission:      Mission,
    participants: list[Player],
    complication: ComplicationCard,
) -> bool:
    """
    Attempt to resolve a mission with the given participants and complication card.

    Checks all preconditions (participant count, tool availability, complication helper
    requirement), computes the final resource requirements, and deducts resources on success.

    Args:
        state:        Current game state (tools, players, pending effects).
        mission:      The mission being attempted.
        participants: Players contributing resources to the mission.
        complication: The complication card drawn for this attempt.

    Returns:
        True if the mission succeeds and resources are deducted, False otherwise.
    """
    # Exact participant count
    if len(participants) != mission.players_count:
        return False

    # Tool availability
    for tool in mission.required_tools:
        if state.tools[tool].damaged:
            return False

    # Night Anxiety: need 1 non-participant, non-exhausted helper
    if complication.requires_extra_helper:
        helpers = [p for p in state.players if p not in participants and not p.is_exhausted]
        if not helpers:
            return False

    requirements = compute_requirements(mission, participants, complication, state)

    success = check_and_contribute(
        participants = participants,
        requirements = requirements,
        max_per_type = complication.max_resource_per_type,
    )

    if success and complication.damages_tool_on_success is not None:
        state.tools[complication.damages_tool_on_success].damaged = True

    return success

In [ ]:
# ── Volcano card effects ───────────────────────────────────────────────────

def apply_volcano_card(
    volcano_card_name: VolcanoCardName,
    state:             GameState,
) -> None:
    """
    Apply the immediate effect of a volcano card drawn after a mission failure.

    Cards with immediate effects (resource loss, exhaustion extension, mission discard)
    are resolved now. Cards with persistent effects are stored in state.pending_volcano_card
    and consumed at the appropriate step of the next round.

    Args:
        volcano_card_name: The name of the drawn volcano card.
        state:             Current game state, mutated in place.
    """
    card = VOLCANO_CARDS[volcano_card_name]

    if card.each_player_loses_resources > 0:
        for player in state.players:
            if card.rich_player_loses_threshold is None or len(player.resources) >= card.rich_player_loses_threshold:
                for _ in range(card.each_player_loses_resources):
                    if player.resources:
                        player.resources.remove(random.choice(player.resources))

    if card.extend_exhaustion_rounds > 0:
        for player in state.players:
            if player.exhausted_until > state.round:
                player.exhausted_until += card.extend_exhaustion_rounds

    if card.discard_mission:
        non_boat_missions = [m for m in state.active_missions if MISSIONS[m].mission_type != MissionType.BOAT]
        candidates = non_boat_missions if non_boat_missions else list(state.active_missions)
        if candidates:
            discarded = random.choice(candidates)
            state.active_missions.remove(discarded)
            
            replacement = draw_mission(state)
            if replacement:
                state.active_missions.append(replacement)

    is_immediate = card.each_player_loses_resources > 0 or card.extend_exhaustion_rounds > 0 or card.discard_mission
    if not is_immediate:
        state.pending_volcano_card = volcano_card_name

In [ ]:
# ── Bonus effect application ───────────────────────────────────────────────

def apply_bonus(
    bonus:        Optional[BonusEffect],
    participants: list[Player],
    mission_name: MissionName,
    state:        GameState,
) -> bool:
    """
    Apply a mission success bonus effect to the game state.

    Handles all bonus types: boat part registration, complication skip, failure protection,
    tool repair, volcano card negation, and resource discounts / gather bonuses for next round.

    Args:
        bonus:        The bonus effect to apply, or None if the mission has no bonus.
        participants: Players who completed the mission (used if the bonus grants no exhaustion).
        mission_name: The completed mission's name (used to register a boat part if applicable).
        state:        Current game state, mutated in place.

    Returns:
        True if participants should skip exhaustion this round, False otherwise.
    """
    if bonus is None:
        return False

    if bonus.boat_part:
        state.boat_parts_built.add(mission_name)

    if bonus.skip_next_complication:
        state.skip_next_complication = True

    if bonus.protect_next_failure:
        state.protect_next_failure = True

    if bonus.repair_tool:
        for tool, tool_state in state.tools.items():
            if tool_state.damaged:
                tool_state.damaged = False
                tool_state.repair_due = None
                break

    if bonus.negates_volcano_card is not None:
        if state.pending_volcano_card == bonus.negates_volcano_card:
            state.pending_volcano_card = None

    if bonus.resource_discount or bonus.resource_discount_any > 0 or bonus.gather_bonus > 0:
        state.pending_bonus = bonus

    return bonus.no_exhaustion

In [ ]:
# ── Agent decisions ────────────────────────────────────────────────────────

def vote_for_mission(player: Player, state: GameState) -> MissionName:
    """
    Return this player's preferred mission, or random if they have no character preference.

    Each character has a priority heuristic (Builder prefers wood-heavy missions,
    Fire Starter prefers fire missions, etc.). When the volcano deck is running low,
    all characters prioritise boat missions regardless of their usual preference.
    If a Panic volcano card is pending, only missions within its participant cap
    are considered.

    Args:
        player: The player casting a vote.
        state:  Current game state (active missions, volcano deck size).

    Returns:
        The preferred MissionName.
    """
    active = state.active_missions
    if state.pending_volcano_card is not None:
        cap = VOLCANO_CARDS[state.pending_volcano_card].max_mission_participants
        if cap is not None:
            valid_missions = [mission_name for mission_name in active if MISSIONS[mission_name].players_count <= cap]
            if valid_missions:
                active = valid_missions
    urgent = len(state.volcano_deck) <= URGENT_VOLCANO_THRESHOLD
    boat_options = [mission_name for mission_name in active if MISSIONS[mission_name].mission_type == MissionType.BOAT]

    if urgent and boat_options:
        return random.choice(boat_options)

    if player.character == Character.BUILDER:
        for mission_name in active:
            if MISSIONS[mission_name].required_resources.get(Resource.WOOD, 0) >= 2:
                return mission_name
    elif player.character == Character.FIRE_STARTER:
        for mission_name in active:
            if MISSIONS[mission_name].mission_type == MissionType.FIRE:
                return mission_name
    elif player.character == Character.COOK:
        for mission_name in active:
            if Tool.VESSEL in MISSIONS[mission_name].required_tools:
                return mission_name
    elif player.character == Character.SAILOR:
        if boat_options:
            return random.choice(boat_options)

    return np.random.choice(active)


def select_mission(state: GameState) -> Optional[MissionName]:
    """
    Aggregate all player votes and return the mission with the most support.

    Players with no preference contribute a random vote. Ties are broken randomly.
    If a Panic volcano card is pending and no active mission fits within the
    participant cap, returns None to signal that the round has no valid mission.

    Args:
        state: Current game state (players, active missions).

    Returns:
        The selected MissionName for this round, or None if no mission fits the Panic cap.
    """
    if state.pending_volcano_card is not None:
        cap = VOLCANO_CARDS[state.pending_volcano_card].max_mission_participants
        if cap is not None:
            valid_missions = [mission_name for mission_name in state.active_missions if MISSIONS[mission_name].players_count <= cap]
            if not valid_missions:
                return None

    votes = Counter(vote_for_mission(player, state) for player in state.players)
    max_votes = max(votes.values())
    top_missions = [mission_name for mission_name, vote_count in votes.items() if vote_count == max_votes]

    return random.choice(top_missions)


def decide_action(
    player:                Player,
    mission:               Mission,
    n_participants_so_far: int,
    state:                 GameState,
) -> PlayerAction:
    """
    Decide what action a player takes this round: participate, gather, or repair.

    Priority order:
    1. Craftsman repairs a damaged tool if they have stone and a slot is available.
    2. Exhausted players always gather.
    3. Players with ≤ 1 resource gather (conservation, unless urgent).
    4. Players avoid piling on when >50% of the group is exhausted (unless urgent).
    5. Players participate if they can cover their share and keep at least 1 resource.
    6. Otherwise gather.

    Args:
        player:                The player deciding.
        mission:               The selected mission for this round.
        n_participants_so_far: Number of players already committed to the mission.
        state:                 Current game state (tools, players, volcano deck).

    Returns:
        The chosen PlayerAction.
    """
    urgent = len(state.volcano_deck) <= URGENT_VOLCANO_THRESHOLD

    # Craftsman: repair if tool damaged and no repair already in progress
    if player.character == Character.CRAFTSMAN and not player.is_exhausted:
        repairable = [
            tool for tool, tool_state in state.tools.items()
            if tool_state.damaged and tool_state.repair_due is None
        ]
        if repairable and Resource.STONE in player.resources:
            return PlayerAction.REPAIR

    if player.is_exhausted:
        return PlayerAction.GATHER

    # Low-pressure conservation: 1 card in hand → gather
    if not urgent and len(player.resources) <= 1:
        return PlayerAction.GATHER

    # Exhaustion spreading: >50% exhausted and mission already has enough participants
    if not urgent:
        exhausted_count = sum(1 for p in state.players if p.is_exhausted)
        if exhausted_count > len(state.players) / 2:
            if n_participants_so_far >= mission.players_count:
                return PlayerAction.GATHER

    # Participation check: can contribute share and keep ≥ 1 card
    total_requirements = sum(mission.required_resources.values())
    share = math.ceil(total_requirements / mission.players_count)
    if len(player.resources) > share:
        return PlayerAction.PARTICIPATE

    return PlayerAction.GATHER


def choose_gather_amount(player: Player) -> int:
    if player.character == Character.GATHERER and not player.is_exhausted and len(player.resources) < 3:
        return 2
    
    return 1

In [ ]:
# ── Round execution ────────────────────────────────────────────────────────

def run_round(state: GameState) -> tuple[bool, bool]:
    """
    Execute one full round of the game.

    Steps in order:
    1. Mission selection — players vote; the winning mission is selected.
       If a Panic volcano card is pending, only missions within its participant
       cap are eligible. If none qualify, the round fails and a volcano card
       is drawn instead.
    2. Player actions — each player decides to participate, gather, or repair.
    3. Complication — a complication card is drawn (Sailor gets best of two on boat missions).
    4. Resolution — mission is attempted; on success bonuses are applied; on failure a
       volcano card is drawn (eruption ends the game immediately).
    5. Exhaustion — participants are exhausted; Ash in the Air adds an extra round.
    6. Gather — gatherers draw resources; Heat Wave cancels gathering.
    7. Mission maintenance — completed mission is replaced from the pool.

    Args:
        state: Current game state, mutated in place.

    Returns:
        A (game_over, won) tuple. game_over is True when the game ends (win or eruption loss).
    """
    state.round += 1
    update_tool_repairs(state)
    refresh_exhaustion(state)

    # Step 1 — Mission selection
    mission_name = select_mission(state)

    if mission_name is None:
        # Panic cap: no active mission fits the participant limit — treat as failure
        state.pending_volcano_card = None
        participants = []
        gatherers = list(state.players)
        success = False
        no_exhaustion = False
        if not state.protect_next_failure:
            volcano_card_name = draw_volcano(state)
            if VOLCANO_CARDS[volcano_card_name].is_eruption:
                return True, False
            if (state.pending_bonus is not None
                    and state.pending_bonus.negates_volcano_card == volcano_card_name):
                state.pending_bonus = None
            else:
                apply_volcano_card(volcano_card_name, state)
        else:
            state.protect_next_failure = False
    else:
        mission = MISSIONS[mission_name]

        # Clear Panic if it was the pending card
        if (state.pending_volcano_card is not None
                and VOLCANO_CARDS[state.pending_volcano_card].max_mission_participants is not None):
            state.pending_volcano_card = None

        # Step 2 — Player actions
        participants: list[Player] = []
        gatherers: list[Player] = []

        for player in state.players:
            action = decide_action(player, mission, len(participants), state)
            if action == PlayerAction.REPAIR:
                repairable = [tool for tool, tool_state in state.tools.items() if tool_state.damaged and tool_state.repair_due is None]
                state.tools[repairable[0]].repair_due = state.round + 2
                player.resources.remove(Resource.STONE)
                gatherers.append(player)
            elif action == PlayerAction.PARTICIPATE:
                if len(participants) < mission.players_count:
                    participants.append(player)
                else:
                    gatherers.append(player)
            else:
                gatherers.append(player)

        # Step 3 — Complication
        if state.skip_next_complication:
            complication = COMPLICATION_CARDS[ComplicationCardName.CALM_BREEZE]
            state.skip_next_complication = False
        elif any(p.character == Character.SAILOR for p in participants) and mission.mission_type == MissionType.BOAT:
            first_complication_name, second_complication_name = draw_complication(state), draw_complication(state)
            first_complication_card, second_complication_card = COMPLICATION_CARDS[first_complication_name], COMPLICATION_CARDS[second_complication_name]
            complication = first_complication_card if first_complication_card.severity <= second_complication_card.severity else second_complication_card
        else:
            complication = COMPLICATION_CARDS[draw_complication(state)]

        # Step 4 — Resolution
        success = resolve_mission(state, mission, participants, complication)

        if success:
            base_points = mission.points
            fire_bonus = (
                any(p.character == Character.FIRE_STARTER for p in participants)
                and mission.mission_type == MissionType.FIRE
            )
            cook_bonus = (
                any(p.character == Character.COOK for p in participants)
                and Tool.VESSEL in mission.required_tools
            )
            point_penalty = VOLCANO_CARDS[state.pending_volcano_card].mission_point_penalty if state.pending_volcano_card is not None else 0

            for player in participants:
                points = base_points + (1 if fire_bonus else 0) + (1 if cook_bonus else 0)
                if point_penalty > 0:
                    points = max(0, points - point_penalty)
                player.score += points

            if point_penalty > 0:
                state.pending_volcano_card = None

            no_exhaustion = apply_bonus(mission.bonus_on_success, participants, mission_name, state)

        else:
            no_exhaustion = False
            if state.protect_next_failure:
                state.protect_next_failure = False
            else:
                volcano_card_name = draw_volcano(state)
                if VOLCANO_CARDS[volcano_card_name].is_eruption:
                    return True, False
                if (state.pending_bonus is not None
                        and state.pending_bonus.negates_volcano_card == volcano_card_name):
                    state.pending_bonus = None
                else:
                    apply_volcano_card(volcano_card_name, state)

        # Step 5 — Exhaustion
        extra_exhaustion = VOLCANO_CARDS[state.pending_volcano_card].extra_exhaustion_rounds if state.pending_volcano_card is not None else 0
        if extra_exhaustion > 0:
            state.pending_volcano_card = None
        if not no_exhaustion:
            apply_exhaustion(participants, state.round, extra_rounds = extra_exhaustion)

    # Step 6 — Gather
    gather_yields_zero = VOLCANO_CARDS[state.pending_volcano_card].gather_yields_zero if state.pending_volcano_card is not None else False
    if gather_yields_zero:
        state.pending_volcano_card = None

    gather_bonus = state.pending_bonus.gather_bonus if state.pending_bonus else 0

    for player in gatherers:
        if gather_yields_zero:
            continue
        base_amount = choose_gather_amount(player)
        total = base_amount + gather_bonus
        for _ in range(total):
            player.resources.append(draw_resource(state))
        if player.character == Character.GATHERER and base_amount == 2:
            apply_exhaustion([player], state.round)

    state.pending_bonus = None

    # Step 7 — Mission maintenance
    if success:
        state.active_missions.remove(mission_name)
        new_mission = draw_mission(state)
        if new_mission:
            state.active_missions.append(new_mission)

    # Win check
    if len(state.boat_parts_built) >= state.boat_parts_required:
        return True, True

    return False, False

In [ ]:
# ── Game runner ────────────────────────────────────────────────────────────

@dataclass
class GameRecord:
    player_count:            int
    characters:              list[Character]
    outcome:                 str
    rounds_played:           int
    final_scores:            dict[Character, int]
    boat_parts_built:        int
    boat_parts_required:     int
    volcano_cards_remaining: int


def run_game(player_count: int, characters: list[Character] = None) -> GameRecord:
    """
    Run a single complete game and return a record of the outcome.

    The game runs for up to 200 rounds. If the round limit is reached without a win
    or eruption, the result is recorded as a loss.

    Args:
        player_count: Number of players (2–4).
        characters:   Optional fixed character list. If None, characters are assigned randomly.

    Returns:
        A GameRecord with the outcome, scores, and game statistics.
    """
    state = init_game(player_count, characters)

    for _ in range(200):
        game_over, won = run_round(state)
        if game_over:
            outcome = "win" if won else "loss"
            break
    else:
        outcome = "loss"

    return GameRecord(
        player_count            = player_count,
        characters              = [p.character for p in state.players],
        outcome                 = outcome,
        rounds_played           = state.round,
        final_scores            = {p.character: p.score for p in state.players},
        boat_parts_built        = len(state.boat_parts_built),
        boat_parts_required     = state.boat_parts_required,
        volcano_cards_remaining = len(state.volcano_deck),
    )


def run_scenario(
    player_count: int,
    n_games:      int,
    base_seed:    Optional[int] = None,
) -> list[GameRecord]:
    """
    Run multiple games with the same player count and collect the results.

    Args:
        player_count: Number of players per game (2–4).
        n_games:      Number of games to simulate.
        base_seed:    If provided, seeds each game deterministically (base_seed + game_index)
                      for reproducible results.

    Returns:
        List of GameRecord, one per game.
    """
    results = []
    for i in range(n_games):
        if base_seed is not None:
            random.seed(base_seed + i)
        results.append(run_game(player_count))
    return results

#### Configure Simulation Scenarios

- Scenarios by player count: 4, 5, 6, 7, 8
- Character composition: random assignment vs. fixed "strong" team vs. fixed "weak" team
- Strategy variants: aggressive (always participate) vs. conservative (gather-heavy)
- Number of runs per scenario: N = 1,000–10,000 (tune based on variance)

In [ ]:
results = {}
for pc in [4, 5, 6, 7, 8]:
    results[pc] = run_scenario(player_count = pc, n_games = 1000, base_seed = 42)

#### Exploratory Simulation Runs

- Single-game trace to verify logic (print round-by-round state)
- Check distribution of game lengths (rounds to win or lose)
- Sanity checks: resource deck exhaustion rate, average volcano cards drawn per game

In [ ]:
import random
import matplotlib.pyplot as plt

# ── Single-game verbose trace ──────────────────────────────────────────────

random.seed(0)
state = init_game(player_count = 5)
print(f"Characters: {[p.character.value for p in state.players]}")
print(f"Boat parts required: {state.boat_parts_required}")
print(f"Active missions: {[m.value for m in state.active_missions]}")
print()

game_over = False
won       = False
while not game_over:
    prev_round     = state.round
    prev_volcano   = len(state.volcano_deck)
    prev_parts     = len(state.boat_parts_built)
    prev_missions  = list(state.active_missions)
    prev_scores    = [p.score for p in state.players]

    game_over, won = run_round(state)

    mission_delta = [m for m in prev_missions if m not in state.active_missions]
    parts_delta   = len(state.boat_parts_built) - prev_parts
    volcano_used  = prev_volcano - len(state.volcano_deck)
    scores        = [p.score for p in state.players]
    score_gains   = [scores[i] - prev_scores[i] for i in range(len(scores))]

    completed = mission_delta[0].value if mission_delta else "—"
    print(
        f"Round {state.round:>2} | mission: {completed:<28} "
        f"| volcano left: {len(state.volcano_deck):>2} (used {volcano_used}) "
        f"| boat: {len(state.boat_parts_built)}/{state.boat_parts_required} "
        f"| scores: {scores} (+{score_gains})"
    )

print()
print(f"Outcome: {'WIN' if won else 'LOSS'} after {state.round} rounds")
print(f"Final scores: { {p.character.value: p.score for p in state.players} }")

# ── Win rates per player count ─────────────────────────────────────────────

player_counts = sorted(results.keys())
win_rates     = [
    sum(1 for r in results[pc] if r.outcome == "win") / len(results[pc]) * 100
    for pc in player_counts
]

fig, axes = plt.subplots(1, 3, figsize = (15, 5))

axes[0].bar([str(pc) for pc in player_counts], win_rates, color = "steelblue")
axes[0].set_xlabel("Player count")
axes[0].set_ylabel("Win rate (%)")
axes[0].set_title("Win rate by player count")
axes[0].set_ylim(0, 100)
for i, v in enumerate(win_rates):
    axes[0].text(i, v + 1, f"{v:.1f}%", ha = "center", fontsize = 9)

# ── Game length distribution (wins only) ─────────────────────────────────

all_lengths = [r.rounds_played for pc in player_counts for r in results[pc] if r.outcome == "win"]
axes[1].hist(all_lengths, bins = 20, color = "seagreen", edgecolor = "white")
axes[1].set_xlabel("Rounds played")
axes[1].set_ylabel("Games")
axes[1].set_title("Game length distribution (wins)")

# ── Avg volcano cards remaining at game end ───────────────────────────────

avg_volcano = [
    sum(r.volcano_cards_remaining for r in results[pc]) / len(results[pc])
    for pc in player_counts
]
axes[2].bar([str(pc) for pc in player_counts], avg_volcano, color = "tomato")
axes[2].set_xlabel("Player count")
axes[2].set_ylabel("Cards remaining (avg)")
axes[2].set_title("Avg volcano cards remaining at end")

plt.tight_layout()
plt.show()

# ── Sanity checks ─────────────────────────────────────────────────────────

print("\nSanity checks")
print("─" * 50)
for pc in player_counts:
    games      = results[pc]
    wins       = [r for r in games if r.outcome == "win"]
    losses     = [r for r in games if r.outcome == "loss"]
    avg_rounds = sum(r.rounds_played for r in games) / len(games)
    avg_vc     = sum(r.volcano_cards_remaining for r in games) / len(games)
    avg_parts  = sum(r.boat_parts_built for r in games) / len(games)
    print(
        f"  {pc}p | wins: {len(wins):>4}/{len(games)} ({len(wins)/len(games)*100:5.1f}%) "
        f"| avg rounds: {avg_rounds:5.1f} "
        f"| avg volcano left: {avg_vc:4.1f} "
        f"| avg boat parts: {avg_parts:.2f}/{games[0].boat_parts_required}"
    )

#### Visualize Outcomes

- Win rate by player count (bar chart with confidence intervals)
- Distribution of game length (histogram)
- Character score distributions (violin or box plot per character)
- Volcano deck depth at game end (how close to eruption)
- Resource economy: average cards in hand per round
- Mission completion heatmap (which missions complete most often)

In [ ]:
# Plots


#### Statistical Analysis

- Bayesian estimation of win rate per scenario using PyMC Beta-Binomial model
- Posterior comparison: does player count 6 win significantly more often than 4?
- Character dominance: Bayesian comparison of individual scores per character
- ArviZ for posterior visualization and credible intervals

In [ ]:
import pymc as pm
import arviz as az

# PyMC models + ArviZ plots

### Results

- Win rates across player counts - is it easier with more players?
- Which characters emerge as score leaders? Is there a dominant strategy?
- Resource bottlenecks: which resource type runs out most often?
- How often does the volcano erupt vs. the group escaping?
- Boat mission difficulty ranking (which cause the most failures?)
- Complication card impact - which ones most often flip a win to a loss?

### Discussion / Further Work

- Is the game balanced as designed, or does it favor a specific player count?
- Suggestions for rule tweaks to improve balance (adjust required boat parts, resource deck size)
- Limitations: heuristic strategies don't capture human creativity or negotiation
- Next steps:
  - Replace rule-based agents with learned strategies (RL or MCTS)
  - Explore asymmetric information (players don't see each other's hands)
  - Sensitivity to rule variants (e.g., exhaustion only on success)
  - Compare simulated win rates against real playtest results

### References / Bibliography

[1] [Schell, J. (2009). Level 16: Game Balance. Game Design Concepts.](https://gamedesignconcepts.wordpress.com/2009/08/20/level-16-game-balance/)

[2] [Chaslot, G., Bakkes, S., Szita, I., & Spronck, P. (2008). Monte-Carlo Tree Search: A New Framework for Game AI. AIIDE.](https://sander.landofsand.com/publications/AIIDE08_Chaslot.pdf)

[3] [StudyGuides.com. (2026). Simulations in Game Theory.](https://studyguides.com/study-methods/overview/clzggcm6j0j8t8cfe9zugo2t8)

[4] [Dubois, E., Barreteau, O., & Souchère, V. (2013). An Agent-Based Model to Explore Game Setting Effects on Attitude Change During a Role Playing Game Session. Journal of Artificial Societies and Social Simulation, 16(1), 2.](https://www.jasss.org/16/1/2.html)

[5] [Yiannakoulias, N., Grignon, M., & Marshall, T. (2024). Parameterizing agent-based models using an online game. Computers, Environment and Urban Systems, 112, 102142.](https://www.sciencedirect.com/science/article/pii/S0198971524000711)

### Appendix

- **Appendix A:** General game rules (overview, resources, mechanics, scoring) - `docs/game_rules.md`
- **Appendix B:** Mission table - `docs/missions.md`
- **Appendix C:** Complication card descriptions - `docs/complication_cards.md`
- **Appendix D:** Volcano card descriptions - `docs/volcano_cards.md`
- **Appendix E:** Character ability reference - `docs/characters.md`
- **Appendix F:** Raw simulation output tables *(to be filled after simulation runs)*